In [2]:
import pygame
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
from PIL import Image
import random
import matplotlib.pyplot as plt
from matplotlib import animation
from collections import deque  ## multiple frame

print(torch.cuda.is_available())  # 應該為 True
print(torch.version.cuda)         # 應該列出 CUDA 版本
print(torch.backends.cudnn.version())  # cuDNN 版本

script_dir = os.path.join(os.getcwd(), 'space_ship_game_RL')
if script_dir not in sys.path:
    sys.path.append(script_dir)

from setting import *
from game import Game

pygame 2.6.1 (SDL 2.28.4, Python 3.12.11)
Hello from the pygame community. https://www.pygame.org/contribute.html
True
11.8
90100


In [3]:
class SpaceShipEnv():
    def __init__(self):
        pygame.init()
        pygame.font.init()

        # 延後畫面初始化，等 render() 時才設置
        self.screen = None
        self.clock = pygame.time.Clock()
        self.fps = FPS

        self.game = Game()

        self.action_space = [0, 1, 2, 3]
        self.observation = self.game.state

        self.in_cooldown = False                 # 自行在 __init__ 加這旗標
        self.cooldown_penalized = True           # 同上

    MAX_ROCK   = 5
    MAX_POWER  = 2
    MAX_SPD_X  = 3
    MAX_SPD_Y  = 10

    def _extract_state(self):
        p = self.game.player.sprite

        # -------- 玩家自身 --------
        px_norm   = p.rect.centerx / WIDTH
        hp_norm   = p.health / 100
        gun_oh    = [1 if i == p.gun - 1 else 0 for i in range(3)]
        cd_norm   = len(p.bullet_timer) / p.bullet_delay  # 0~1

        # -------- 石頭 (最近 10 顆) --------
        rocks = sorted(self.game.rocks,
                       key=lambda r: (r.rect.y - p.rect.y)**2 + (r.rect.x - p.rect.x)**2
                       )[:self.MAX_ROCK]

        rock_feats = []
        for r in rocks:
            dx = (r.rect.centerx - p.rect.centerx) / WIDTH
            dy = (r.rect.centery - p.rect.centery) / HEIGHT
            vx = r.speedx / self.MAX_SPD_X          # 約 -1~1
            vy = r.speedy / self.MAX_SPD_Y          # 約 0.2~1
            rr = r.radius / 40.0
            rock_feats += [dx, dy, vx, vy, rr]
        rock_feats += [0]*(self.MAX_ROCK*5 - len(rock_feats))  # padding

        # -------- 道具 (最近 2 顆) --------
        powers = sorted(self.game.powers,
                        key=lambda pw: abs(pw.rect.y - p.rect.y))[:self.MAX_POWER]

        power_feats = []
        for pw in powers:
            dx = (pw.rect.centerx - p.rect.centerx) / WIDTH
            dy = (pw.rect.centery - p.rect.centery) / HEIGHT
            tp = 1 if pw.type == 'shield' else -1   # shield=+1, gun=-1
            power_feats += [dx, dy, tp]
        power_feats += [0]*(self.MAX_POWER*3 - len(power_feats))

        # -------- 組合 --------
        state_vec = np.array(
            [px_norm, hp_norm] + gun_oh + [cd_norm] +
            rock_feats + power_feats,
            dtype=np.float32
        )
        return state_vec

    # ---- 超參數（一次集中管理） ----
    ALPHA_HIT      = 1.4           # 其實已併入 delta_score，可設 1
    LAMBDA_COLL    = 1.2           # 撞擊倍率
    # ---- 冷卻期 shaping ----
    MISS_SHOT_PENALTY = -1     # 空槍
    COOLDOWN_BONUS    = 0.5     # 射後等待

    def step(self, action):
        # ----- 0. 撞擊前狀態 -----
        player       = self.game.player.sprite
        ready_before = player.bullet_ready
        was_shooting  = (action == 1)
        hp_before    = player.health    
        score_before = self.game.score
        
        # ----- 1. 更新遊戲 -----
        self.game.update(action)

        ready_after  = player.bullet_ready          # ★update 後
        fired_now    = was_shooting and ready_before  # 這幀真的發射

        if self.screen is None:
            self.game.draw()
        else:
            self.game.draw(self.screen)
            self.clock.tick(self.fps)

        # ----- 2. 計算 reward -----
        reward = -0.3       # 基礎時間懲罰

        # (a) 擊破石頭
        delta_score = self.game.score - score_before
        reward += self.ALPHA_HIT * delta_score

        # (b) 撞擊懲罰（半徑 × 剩餘血量因子）
        if self.game.is_collided:
            hp_after  = self.game.player.sprite.health
            radius    = hp_before - hp_after                # = damage = radius
            factor    = 2 - hp_after / 100                  # 滿血1 → 殘血2
            penalty   = self.LAMBDA_COLL * radius * factor
            reward   -= penalty

        # (c) 撿道具（依前述 shield +γ·hp_gain, gun +5）
        if self.game.is_power:
            hp_gain = self.game.player.sprite.health - hp_before
            reward += hp_gain                        # shield +20
            if hp_gain == 0:                               # gun
                reward += 12


        # -- 冷卻開始：扣一次 --
        if fired_now:
            self.in_cooldown = True                 # 自行在 __init__ 加這旗標
            self.cooldown_penalized = False         # 同上

        if was_shooting and not ready_before:       # 狂按但未射出
            if not self.cooldown_penalized:
                reward += self.MISS_SHOT_PENALTY         # 只扣一次
                self.cooldown_penalized = True

        # -- 冷卻結束：加一次 --
        if self.in_cooldown and ready_after:
            reward += self.COOLDOWN_BONUS
            self.in_cooldown = False                # 重置旗標

        # ----- 3. 其餘回傳 -----
        done  = (not self.game.running) or (self.game.score >= 10000)
        info  = self.game.score
        state = self._extract_state()

        return state, reward, done, info

    def reset(self):
        self.game = Game()
        self.in_cooldown = False          # ★
        self.cooldown_penalized = True    # ★
        return self._extract_state()

    def render(self):
        if self.screen is None:
            self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
            pygame.display.set_caption("SpaceShip RL Environment")

    def close(self):
        pygame.quit()


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
class MLPDQN(nn.Module):
    def __init__(self, input_dim, num_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions)
        )
    def forward(self, x):
        return self.net(x)

In [9]:
env = SpaceShipEnv()
state_np = env.reset()                               # numpy.ndarray
input_dim  = state_np.shape[0]
num_actions = 4

model = MLPDQN(input_dim, num_actions).to(device)
ckpt = torch.load('checkpoint_ep1000.pth', map_location=device)
model.load_state_dict(ckpt['policy_net'])
model.eval()


MLPDQN(
  (net): Sequential(
    (0): Linear(in_features=37, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

In [10]:
# Visualization of trained agent
env.render()
state = state_np
done = False
total_reward = 0
frames = []

while not done:
    state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(device)
    with torch.no_grad():
        action = model(state_tensor).argmax(dim=1).item()
        
    # ---- 執行一步 ----
    next_state, reward, done, score = env.step(action)
    total_reward += reward
    state = next_state    
    
    # 把畫面抓下來（RGB）
    surface = pygame.display.get_surface()
    frame = np.transpose(pygame.surfarray.array3d(surface), (1, 0, 2))     # pygame 是 x,y → imageio 是 y,x
    frames.append(frame)

    pygame.event.pump() # 避免無回應

print(f"reward: {total_reward}, score: {score}")
print(len(frames))
env.close()

reward: 1454.5800000000906, score: 1836
2891


In [8]:
import imageio

video_path = "space_ship_run_rl_best.mp4"

imageio.mimsave(video_path, frames, fps=60, quality=9)
print(f"Saved gameplay video to: {video_path}")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (250, 300) to (256, 304) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved gameplay video to: space_ship_run_rl_best.mp4
